# GOL651 – 01 · NumPy, Pandas et Matplotlib
*GOL651 – 01 · NumPy, Pandas and Matplotlib*

**Objectifs.** Calculer avec des tableaux NumPy, analyser un fichier CSV avec Pandas et produire des graphiques avec Matplotlib.
Les données (`data/datasets/shipments.csv`) décrivent 40 expéditions fictives d'une entreprise de distribution au Québec.

*Goals: compute with NumPy arrays, analyse a CSV file with Pandas and plot with Matplotlib. The data describe 40 fictional shipments of a Québec distribution company.*

> Exécutez les cellules dans l'ordre avec **Maj + Entrée**. / *Run the cells in order with **Shift + Enter**.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
print("NumPy", np.__version__, "| Pandas", pd.__version__)

In [ ]:
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Folder that contains data/datasets (works inside Docker and on a laptop)."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "datasets").is_dir():
            return folder
    raise FileNotFoundError("Project root not found: open this notebook from the course folder")

ROOT = find_project_root()
DATA = ROOT / "data"
print("Project folder:", ROOT)

## 1. NumPy : calcul vectoriel / *vectorized computation*

La **quantité économique de commande** (modèle de Wilson) minimise la somme des coûts de commande et de stockage :

$$Q^{*} = \sqrt{\frac{2\,D\,S}{H}}$$

où $D$ est la demande annuelle, $S$ le coût par commande et $H$ le coût de stockage annuel par unité.
NumPy applique la formule à **tous les produits à la fois**, sans boucle.

*The economic order quantity minimizes ordering + holding costs. NumPy evaluates it for all products at once, without a loop.*

In [ ]:
products = np.array(["PAL-001", "GAN-080", "CAS-081", "ETQ-040"])
D = np.array([4_000, 6_000, 1_200, 2_500])      # annual demand (units)
S = np.array([45.0, 30.0, 30.0, 25.0])          # cost per order ($)
H = np.array([2.20, 0.80, 3.10, 1.10])          # holding cost ($/unit/year)

Q_star = np.sqrt(2 * D * S / H)                  # element-wise operations
orders_per_year = D / Q_star
total_cost = (D / Q_star) * S + (Q_star / 2) * H

for p, q, n, c in zip(products, Q_star, orders_per_year, total_cost):
    print(f"{p:<8} Q* = {q:7.1f} units   {n:5.1f} orders/year   cost {c:8.2f} $")

## 2. Pandas : lire et explorer un CSV / *read and explore a CSV*

In [ ]:
shipments = pd.read_csv(DATA / "datasets" / "shipments.csv", parse_dates=["ship_date"])
print(shipments.shape)
shipments.head()

In [ ]:
shipments.describe(include="number").round(2)

### Agrégations / *aggregations*
Coût moyen et poids total par transporteur, puis tableau croisé destination × transporteur.

*Average cost and total weight per carrier, then a destination × carrier pivot table.*

In [ ]:
by_carrier = (
    shipments.groupby("carrier")
    .agg(shipments=("shipment_id", "count"),
         total_weight_kg=("weight_kg", "sum"),
         avg_cost_cad=("cost_cad", "mean"))
    .round(1)
    .sort_values("avg_cost_cad")
)
by_carrier

In [ ]:
pd.pivot_table(shipments, index="destination_city", columns="carrier",
               values="cost_cad", aggfunc="sum", fill_value=0).round(0)

## 3. Régression linéaire simple avec NumPy / *simple linear regression*

On estime le coût d'une expédition en fonction de son poids : $\hat{c} = \beta_0 + \beta_1\,w$.

*We estimate the cost of a shipment from its weight.*

In [ ]:
beta1, beta0 = np.polyfit(shipments["weight_kg"], shipments["cost_cad"], deg=1)
r = np.corrcoef(shipments["weight_kg"], shipments["cost_cad"])[0, 1]
print(f"cost ≈ {beta0:.2f} + {beta1:.4f} × weight   (R² = {r**2:.3f})")

## 4. Matplotlib : graphiques / *charts*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(shipments["weight_kg"], shipments["cost_cad"], alpha=0.7, label="shipments")
w = np.linspace(0, shipments["weight_kg"].max(), 50)
ax1.plot(w, beta0 + beta1 * w, color="black", label="linear fit")
ax1.set(title="Cost vs weight", xlabel="Weight (kg)", ylabel="Cost ($)")
ax1.legend()

monthly = shipments.set_index("ship_date")["cost_cad"].resample("MS").sum()
ax2.bar(monthly.index.strftime("%Y-%m"), monthly.values)
ax2.set(title="Transport cost per month", ylabel="Cost ($)")
ax2.tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()

### À vous / *Your turn*
1. Quel transporteur a le meilleur coût par kilogramme ? / *Which carrier has the best cost per kilogram?*
2. Refaites la régression pour un seul transporteur. / *Redo the regression for a single carrier.*